# AMPR Phase 3 — Train MF Branch on Kaggle 2×T4

Run on **Kaggle Notebook** with 2×T4 GPU (~2-3h).

Required Kaggle Datasets (attach before running):
- `ampr-phase3-embeddings` — ESM-2 HDF5, PPI npy files
- `ampr-pdbch` — PDBch labels, DAG matrices, splits, protein_order, GO embeddings, cmap_all.h5

Expected: `val Fmax (DAG-prop)` ≥ 0.30 after 10 epochs (vs AMPR v1 baseline 0.158).

In [ ]:
import subprocess, os
os.makedirs('/kaggle/working/datn', exist_ok=True)
!git clone https://github.com/hungithust/datn_protein_function /kaggle/working/datn
%cd /kaggle/working/datn
!pip install -q transformers==4.41.2 obonet biopython==1.84 h5py pyyaml tqdm

In [ ]:
import os
os.makedirs('data/embeddings', exist_ok=True)
os.makedirs('data/contact_maps', exist_ok=True)
os.makedirs('data/pdbch', exist_ok=True)

# Symlink embeddings
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-phase3-embeddings/esm2_residue.h5 data/embeddings/esm2_residue.h5
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-phase3-embeddings-2/ppi_deepgo.npy data/embeddings/ppi_deepgo.npy
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-phase3-embeddings-2/ppi_deepgo_mask.npy data/embeddings/ppi_deepgo_mask.npy

# Symlink PDBch artifacts
!ln -sf /kaggle/input/datasets/hungnguyenviet04/cmap-all/cmap_all.h5 data/contact_maps/cmap_all.h5
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-pdbch-phase0/labels_mf.npy data/pdbch/labels_mf.npy
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-pdbch-phase0/dag_matrix_mf.npy data/pdbch/dag_matrix_mf.npy
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-pdbch-phase0/splits.json data/pdbch/splits.json
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-pdbch-phase0/protein_order.json data/pdbch/protein_order.json
!ln -sf /kaggle/input/datasets/hungnguyenviet04/ampr-pdbch-phase0/go_emb_mf.npy data/embeddings/go_emb_mf.npy

print('Symlinks ready.')

In [ ]:
# Verify GPU setup
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## 🔍 VERIFY INPUTS (chạy trước khi train)

Kiểm tra **độ khớp giữa mọi input với nhau và với config/code** để loại trừ nguyên nhân "model không học" do dữ liệu sai khớp:

- **Alignment theo protein:** `protein_order` ↔ `labels` rows ↔ `ppi_emb` rows ↔ `ppi_mask` (nếu lệch → mỗi protein bị gán nhãn của protein khác → model không thể học).
- **Alignment theo GO-term:** `labels` cols ↔ `dag_matrix` ↔ `go_emb` rows = `n_terms` (489).
- **Khớp với config:** `seq.d_model` ↔ chiều ESM-2, `ppi.in_dim` ↔ chiều PPI.
- **HDF5 coverage:** mọi protein trong split đều có ESM-2 + cmap; chiều residue và L có khớp nhau không.
- **True-Path-Rule:** orientation của `dag_matrix` có khớp giả định trong `loss.py` không.

3 cell dưới chạy **theo thứ tự từ trên xuống** (cell sau dùng biến của cell trước). Copy toàn bộ output paste lại cho mình.

In [ ]:
# === VERIFY 1/3: config + static arrays (protein/term alignment, dtypes, config match) ===
import json, yaml, numpy as np

cfg = yaml.safe_load(open('configs/mf_v3.yaml'))
N_TERMS     = cfg['n_terms']
SEQ_DIM_CFG = cfg['model']['seq']['d_model']
PPI_DIM_CFG = cfg['model']['ppi']['in_dim']
d = cfg['data']
print(f"[CFG] branch={cfg['branch']} n_terms={N_TERMS} seq.d_model={SEQ_DIM_CFG} ppi.in_dim={PPI_DIM_CFG}")
print(f"[CFG] loss={cfg['training']['loss_type']} lr={cfg['training']['lr']} "
      f"asl_gamma_neg={cfg['training'].get('asl_gamma_neg')} asl_clip={cfg['training'].get('asl_clip')} "
      f"lambda_dag={cfg['training'].get('lambda_dag')}")

# protein_order -> list (replicate dataset.py logic exactly)
order = json.loads(open(d['protein_order']).read())
if isinstance(order, dict):
    order = [k for k, _ in sorted(order.items(), key=lambda kv: kv[1])]
N = len(order)
prot2idx = {p: i for i, p in enumerate(order)}

labels   = np.load(d['labels'])
ppi      = np.load(d['ppi_emb'])
ppi_mask = np.load(d['ppi_mask'])
dag      = np.load(d['dag_matrix'])
go_emb   = np.load(d['go_emb'])

results = []
def check(name, cond, detail=""):
    results.append(bool(cond)); print(f"[{'PASS' if cond else 'FAIL'}] {name}  {detail}")

print("\n-- shapes / dtypes --")
print(f"protein_order N = {N}  (unique={len(set(order))})")
print(f"labels    {labels.shape} {labels.dtype}")
print(f"ppi_emb   {ppi.shape} {ppi.dtype}")
print(f"ppi_mask  {ppi_mask.shape} {ppi_mask.dtype}")
print(f"dag       {dag.shape} {dag.dtype}")
print(f"go_emb    {go_emb.shape} {go_emb.dtype}")

print("\n-- ALIGNMENT theo protein (row index phải khớp protein_order) --")
check("protein_order ids unique", len(set(order)) == N)
check("labels rows == N",      labels.shape[0] == N,   f"{labels.shape[0]} vs {N}")
check("ppi rows == N",         ppi.shape[0] == N,      f"{ppi.shape[0]} vs {N}")
check("ppi_mask len == N",     ppi_mask.shape[0] == N, f"{ppi_mask.shape[0]} vs {N}")

print("\n-- ALIGNMENT theo GO-term (cols phải = n_terms) --")
check("labels cols == n_terms", labels.shape[1] == N_TERMS, f"{labels.shape[1]} vs {N_TERMS}")
check("dag square n_terms",     dag.shape == (N_TERMS, N_TERMS), f"{dag.shape}")
check("go_emb rows == n_terms", go_emb.shape[0] == N_TERMS, f"{go_emb.shape[0]} vs {N_TERMS}")

print("\n-- KHỚP CONFIG (chiều input vs model config) --")
check("ESM-2 d_model checked in cell 3/3", True, "(see next cell for esm2 residue dim)")
check("ppi.in_dim == ppi_emb dim", ppi.shape[1] == PPI_DIM_CFG, f"{ppi.shape[1]} vs {PPI_DIM_CFG}")

print("\n-- LABEL semantics --")
check("labels binary {0,1}", set(np.unique(labels).tolist()).issubset({0, 1}),
      f"unique[:5]={np.unique(labels)[:5]}")
pos = labels.sum(1)
empty_terms = int((labels.sum(0) == 0).sum())
print(f"pos/protein: min={pos.min():.0f} mean={pos.mean():.2f} max={pos.max():.0f}")
print(f"proteins with 0 labels: {int((pos == 0).sum())}  | terms with 0 positives: {empty_terms}")
print(f"global positive rate: {labels.mean():.4f}")

print("\n-- PPI mask --")
print(f"ppi_mask True (có PPI): {int(ppi_mask.astype(bool).sum())}/{N} "
      f"({ppi_mask.astype(bool).mean()*100:.1f}%)")
# với protein không có PPI, vector ppi nên ~0
no_ppi = ~ppi_mask.astype(bool)
if no_ppi.any():
    print(f"  |ppi| khi mask=False: mean_abs={np.abs(ppi[no_ppi]).mean():.4f} (kỳ vọng ~0)")

print("\n-- go_emb / dag stats --")
print(f"go_emb row_norm: mean={np.linalg.norm(go_emb,axis=1).mean():.3f} "
      f"max={np.linalg.norm(go_emb,axis=1).max():.3f} abs_max={np.abs(go_emb).max():.3f}")
print(f"  (norm lớn >~20 => head bio dễ bão hoà logit)")
check("dag binary {0,1}", set(np.unique(dag).tolist()).issubset({0, 1}), f"unique[:5]={np.unique(dag)[:5]}")
print(f"dag edges(nonzero)={int(dag.sum())}  diag_sum={int(np.trace(dag))}")

print(f"\n===== VERIFY 1/3: {sum(results)}/{len(results)} PASS =====")

In [ ]:
# === VERIFY 2/3: HDF5 contents + splits coverage + ESM-2/cmap L-alignment ===
# (dùng biến cfg, d, prot2idx, SEQ_DIM_CFG... từ cell VERIFY 1/3)
import h5py, json, numpy as np

results2 = []
def check2(name, cond, detail=""):
    results2.append(bool(cond)); print(f"[{'PASS' if cond else 'FAIL'}] {name}  {detail}")

splits = json.loads(open(d['splits']).read())
print(f"[SPLITS] keys = {list(splits.keys())}")
for k, v in splits.items():
    print(f"  {k}: {len(v)} ids")

print("\n-- split ids ⊆ protein_order --")
for k in ['train', 'valid']:
    miss = [p for p in splits.get(k, []) if p not in prot2idx]
    check2(f"'{k}' ⊆ protein_order", len(miss) == 0, f"missing={len(miss)} e.g.{miss[:3]}")

with h5py.File(d['esm2_h5'], 'r') as fe, h5py.File(d['cmap_h5'], 'r') as fc:
    esm_keys = set(fe.keys()); cm_keys = set(fc.keys())
    print(f"\n[H5] esm2 keys={len(esm_keys)}  cmap keys={len(cm_keys)}")

    print("\n-- coverage: mọi protein trong split phải có esm2 + cmap --")
    for k in ['train', 'valid']:
        ids = splits.get(k, [])
        no_e = [p for p in ids if p not in esm_keys]
        no_c = [p for p in ids if p not in cm_keys]
        check2(f"'{k}' all have esm2", len(no_e) == 0, f"missing={len(no_e)} e.g.{no_e[:3]}")
        check2(f"'{k}' all have cmap", len(no_c) == 0, f"missing={len(no_c)} e.g.{no_c[:3]}")
        kept = [p for p in ids if p in prot2idx and p in esm_keys and p in cm_keys]
        print(f"  '{k}': dataset thực tế giữ {len(kept)}/{len(ids)} proteins")

    print("\n-- per-protein sample: esm2 (L,D) vs cmap (L,L) --")
    sample = [p for p in splits['train'] if p in esm_keys and p in cm_keys][:6]
    Lmis = 0
    for p in sample:
        r = fe[p][:]; c = fc[p][:]
        ok = (r.shape[0] == c.shape[0])
        Lmis += (0 if ok else 1)
        print(f"  {p}: esm2={r.shape} cmap={c.shape} L_match={ok} "
              f"cmap[min={c.min():.1f},max={c.max():.1f}]")
    if sample:
        r0 = fe[sample[0]][:]; c0 = fc[sample[0]][:]
        check2("esm2 residue dim == config seq.d_model",
               r0.shape[1] == SEQ_DIM_CFG, f"{r0.shape[1]} vs {SEQ_DIM_CFG}")
        check2("cmap vuông (LxL)", c0.shape[0] == c0.shape[1], f"{c0.shape}")
        check2("esm2 L == cmap L (sample)", Lmis == 0, f"mismatch={Lmis}/{len(sample)}")
        # cmap nên là khoảng cách Å (>=0, có giá trị < threshold 10 để tạo cạnh)
        thr = cfg['model']['gnn']['cmap_threshold']
        frac_edge = float((c0 < thr).mean())
        check2(f"cmap có cạnh < threshold={thr}", 0.0 < frac_edge < 1.0,
               f"frac<thr={frac_edge:.3f} (0 hoặc 1 => cmap sai đơn vị/giá trị)")

print(f"\n===== VERIFY 2/3: {sum(results2)}/{len(results2)} PASS =====")

In [ ]:
# === VERIFY 3/3: DAG orientation vs loss.py + label↔DAG↔go_emb cùng term-order ===
# (dùng labels, dag, go_emb, d từ cell VERIFY 1/3)
import numpy as np, json
from pathlib import Path

results3 = []
def check3(name, cond, detail=""):
    results3.append(bool(cond)); print(f"[{'PASS' if cond else 'FAIL'}] {name}  {detail}")

L = labels.astype(np.float32)

# True-Path-Rule: child positive => ancestor positive.
# loss.py: violation = relu(probs[i]-probs[j]) * dag[i,j]  => giả định dag[i,j]=1 nghĩa là
#   i là CHILD, j là PARENT (prob con <= prob cha). Orientation ĐÚNG = cái có ÍT vi phạm hơn.
viol_A = float(((L @ dag) * (1.0 - L)).sum())          # A: dag[child,parent]  (khớp loss.py)
viol_B = float(((L @ dag.T) * (1.0 - L)).sum())        # B: dag[parent,child]  (chuyển vị)
tot_pos = float(L.sum())
print("-- True-Path-Rule (đếm vi phạm con-dương-nhưng-cha-âm) --")
print(f"  orientation A  dag[child,parent] (KHỚP loss.py): violations = {viol_A:.0f}")
print(f"  orientation B  dag[parent,child] (chuyển vị)    : violations = {viol_B:.0f}")
# Chỉ cần A < B là orientation khớp loss.py (A càng gần 0 càng tốt)
check3("DAG orientation khớp loss.py (A < B)", viol_A < viol_B, f"A={viol_A:.0f} < B={viol_B:.0f}")
print(f"  [info] vi phạm ở orientation đúng = {viol_A:.0f} "
      f"(~{100*viol_A/max(tot_pos,1):.2f}% số nhãn dương) — nên ~0 nếu labels đã propagate đủ lên DAG")

# go_emb có khớp term-order của labels không?
print("\n-- term-order labels ↔ go_emb ↔ dag --")
print(f"  labels cols={labels.shape[1]}  dag={dag.shape}  go_emb rows={go_emb.shape[0]}")
gt_path = Path('data/pdbch/go_terms_mf.json')
if gt_path.exists():
    terms = json.loads(gt_path.read_text())
    if isinstance(terms, dict):
        terms = [t for t, _ in sorted(terms.items(), key=lambda kv: kv[1])]
    print(f"  go_terms_mf.json: {len(terms)} terms; head={terms[:3]}")
    check3("go_terms list len == n_terms", len(terms) == labels.shape[1], f"{len(terms)}")
else:
    print("  (không thấy go_terms_mf.json — chỉ verify số chiều; đảm bảo go_emb_mf.npy build CÙNG thứ tự term với labels_mf.npy)")

row_std = go_emb.std(axis=1)
dup = (np.unique(go_emb, axis=0).shape[0] < go_emb.shape[0])
check3("go_emb mỗi term khác nhau (không trùng/0)", (row_std > 1e-6).all() and not dup,
       f"rows_zero_std={(row_std<=1e-6).sum()} has_dup={dup}")

# Mất cân bằng lớp (driver chính của collapse với ASL)
print("\n-- Class imbalance (driver của collapse) --")
print(f"  positive rate = {labels.mean():.4f}  ({labels.mean()*100:.2f}%)  "
      f"=> ~{1/labels.mean():.0f} âm / 1 dương")
print(f"  proteins 0-label = {int((labels.sum(1)==0).sum())} ({(labels.sum(1)==0).mean()*100:.1f}%)")

print(f"\n===== VERIFY 3/3: {sum(results3)}/{len(results3)} PASS =====")
print("\n>>> Input khớp nhau => vấn đề ở TRAINING DYNAMICS (positive rate cực thấp ở trên).")

In [ ]:
# Train MF v3 (BCE + pos_weight + grad_clip) — fix dead-gradient của ASL.
# Theo dõi [DIAG]: grad_norm KHÔNG về 0, probs mean > 0, cross_protein_std tăng,
# val_Fmax bứt khỏi 0.0209. Chỉ cần 2-3 epoch đầu để biết đã thoát collapse.
!python main.py --config configs/mf_v3_bce.yaml 2>&1 | tee /kaggle/working/mf_v3_bce_train.log

In [ ]:
# Archive checkpoint + log
import shutil
shutil.copy('checkpoints/mf_v3_bce/best.pt', '/kaggle/working/mf_v3_bce_best.pt')
print('Training complete. Checkpoint saved to /kaggle/working/')